# Creating Vanilla Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [1]:
import pickle
import random
from hashlib import sha256
from tqdm import tqdm
from math import pi, sqrt, e, log
import csv
import time

## Table Parameters

In [2]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

In [3]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)


## Hash and Reduction Functions

In [4]:
# Hash function
def H(x):
	return int.from_bytes(sha256(x.to_bytes(8)).digest())

# Reduction function
# currently mod but should change to murmurhash in future
def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	return (y + i + ell*t) % N

## Building Vanilla Table

In [5]:
# take in m_0 and t as parameters - how many chains to start with and how long to make the chains
# store the table as a dictionary of endpoint:startpoint pairs (rather than sp:ep for easier lookup later)
# store in a pickle file

def build_vanilla_table(t, alpha, startpoints):
    # parameters to store how many hashes and reductions are done, as well as actual time it takes to make this table
    hashes = 0
    reductions = 0
    duration = 0

    # start monitoring time
    start = time.perf_counter()

    # store table in dictionary
    table = {}

    # for each startpoint
    for sp in tqdm(startpoints):
        current_point = sp  # keep track of current point in chain -  we want to store startpoint later

        # create the chain
        for i in range(t):
            # hash then reduce the value
            current_point = r(H(current_point), i)
            hashes += 1
            reductions += 1

        # check if there wasn't a chain merge (not in a value stored already) - if not then store in table
        if current_point not in table:
            table[current_point] = sp

    # finished building
    duration = time.perf_counter() - start

    # store the table as a pickle file
    with open(f'constructed_tables/vanilla_table_alpha_{alpha}_t_{t}.pkl', 'wb') as f:
        pickle.dump(table, f)

    return table, hashes, reductions, duration

## Searching the Table

### Function to help hash and reduce accordingly to continue search

In [6]:
# function to continue search
# we need to take in what column we're at (c), our key (y), # columns (t), current total of hashes and reductions
def continue_search(y, t, c, hashes, reductions):
    # reduce c by 1 to move to the previous column
    c -= 1
    # number of columns between current column and end 
    # diff = t - c  
    # reduce y by the new c index
    x = r(y, c)
    reductions += 1
    # then hash and reduce however many times to move back through columns
    for i in range((t - c) - 1, 0, -1):    # decrease difference by 1 (as we already reduced by current diff index) then continuously reduce by 1
        x = r(H(x), t-i)
        hashes += 1
        reductions += 1

    # return x, c, hashes, reductions
    return x, c, hashes, reductions

### Searching vanilla table function

In [7]:
def search_vanilla_table(y, t, table):
    # keep track of hashes and reductions
    cols_searched = 0
    hashes = 0
    reductions = 0
    false_alarms = 0
    false_alarm_cols = []
    duration = 0

    # start monitoring time
    start = time.perf_counter()

    # keep track of column we're in 
    c = t - 1
    # reduce (r_t-1) the hash then compare in the table 
    x = r(y, c)
    reductions += 1

    # while we haven't reached the end of our chain
    while c > -1:
        # if there is a match in the keys (our endpoints), regenerate chain until we find the key
        if (x in table.keys()):
            point = table[x]   # search for endpoint in table and get startpoint
            # regenerate chain until column c - we are now in the column before the match
            for i in range(c):
                point = r(H(point), i)
                hashes += 1
                reductions += 1
            
            # hash the point - if it is a match we have found our preimage 
            if H(point) == y:
                hashes += 1
                duration = time.perf_counter() - start
                return True, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration
            
            # if that didn't work then we ran into a false alarm
            else:
                false_alarms += 1
                false_alarm_cols.append(c)
                # continue the search
                x, c, hashes, reductions = continue_search(y, t, c, hashes, reductions)
 
        # if we didn't find a match in endpoints, we need to restart the search
        else:
            # hash and reduce the ciphertext accordingly
            x, c, hashes, reductions = continue_search(y, t, c, hashes, reductions)

        cols_searched += 1

    # finished search so stop recording time
    duration = time.perf_counter() - start

    # We have searched all columns - return -1
    return False, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration

## Get Results

### Precomputation Phase - Build the Table

In [8]:
# either build or load table
def get_vanilla_table():
    # try loading table from pickle file
    try:
        with open(f'constructed_tables/vanilla_table_alpha_{alpha}_t_{t}.pkl', 'rb') as f:
            table = pickle.load(f)

    # if no pickle file found, build the table
    except FileNotFoundError:
        table = build_vanilla_table(t, alpha, startpoints)

    return table

### Get data when building tables

In [9]:
# set up loop to build different tables with different cost factors
# create the file and add header - for building the table
with open("results/building_vanilla_table_results.csv", "w", newline="") as fbuilding:
    building_writer = csv.writer(fbuilding)
    building_writer.writerow(["actual_mt", "hashes", "reductions", "duration"])  # headers
    

    # build vanilla table
    table, hashes, reductions, duration = build_vanilla_table(t, alpha, startpoints)

    # store results in csv file
    # ["actual_mt", "hashes", "reductions", "duration"]
    building_writer.writerow([ len(table), hashes, reductions, duration])

100%|██████████| 32510/32510 [00:04<00:00, 7941.16it/s]


### Get data when searching tables

In [10]:
# store how long search takes
time_search = 0


# load the table

# load the data to search - to_search
with open(f'to_search_N_{nlabel}.pkl', 'rb') as f:
    to_search = pickle.load(f)

# make a csv file to store results
with open(f"results/searching_vanilla_table_results.csv", "w", newline="") as fsearching:
    searching_writer = csv.writer(fsearching)
    searching_writer.writerow(["point", "found", "hashes", "reductions", "false_alarms", "false_alarm_cols", "cols_searched", "duration"])  # headers
    
    # start bulk search time monitoring
    start = time.perf_counter()

    # search each point in to_search
    for point in to_search:
        y = H(point)
        found, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration = search_vanilla_table(y, t, table)
        searching_writer.writerow([point, found, hashes, reductions, false_alarms, false_alarm_cols, cols_searched, duration])

    # stop search time monitoring
    time_search = time.perf_counter() - start

print(time_search)
        

175.7124419000029
